# Power System Load Type Classification - Assignment 2

This notebook builds a machine learning model to predict the `Load_Type` of a power system (`Light_Load`, `Medium_Load`, or `Maximum_Load`) from 15-minute measurements and their timestamps.

The final model is evaluated on the last month of data (December 2018), which is never used for training or model selection.

The notebook has been cleaned up to make the workflow more reproducible, easier to maintain, and better suited to real model evaluation.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.imputation import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", __import__("sklearn").__version__)
print("Matplotlib:", plt.matplotlib.__version__)

## 1. Load the dataset

We keep an untouched raw copy of the data so that changes made during cleaning remain transparent and auditable.

In [ ]:
DATA_PATH = Path("data") / "load_data.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Make sure the 'data' folder is in the same directory as this notebook."
    )

df_raw = pd.read_csv(DATA_PATH)

print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")
df_raw.head()

## 2. Rename columns to a more usable format

This reduces confusion caused by punctuation, spaces, and inconsistent naming.

In [ ]:
column_names = {
    "Date_Time": "date_time",
    "Usage_kWh": "usage_kwh",
    "Lagging_Current_Reactive.Power_kVarh": "lagging_reactive_kvarh",
    "Leading_Current_Reactive_Power_kVarh": "leading_reactive_kvarh",
    "CO2(tCO2)": "co2_tco2",
    "Lagging_Current_Power_Factor": "lagging_power_factor",
    "Leading_Current_Power_Factor": "leading_power_factor",
    "NSM": "nsm",
    "Load_Type": "load_type",
}

missing_cols = set(column_names) - set(df_raw.columns)
assert not missing_cols, f"Expected columns not found: {missing_cols}"

df = df_raw.rename(columns=column_names).copy()
df.columns.tolist()

## 3. Basic data quality checks

Power factor values should stay between 0 and 100. NSM must reflect the time of day and should be a multiple of 900 seconds (15 minutes).

In [ ]:
numeric_cols = [
    "usage_kwh", "lagging_reactive_kvarh", "leading_reactive_kvarh", "co2_tco2",
    "lagging_power_factor", "leading_power_factor", "nsm",
]

quality_report = pd.DataFrame({
    "negative_values": (df[numeric_cols] < 0).sum(),
    "max_value": df[numeric_cols].max(),
})
quality_report.loc["lagging_power_factor", "values_above_100"] = (df["lagging_power_factor"] > 100).sum()
quality_report.loc["leading_power_factor", "values_above_100"] = (df["leading_power_factor"] > 100).sum()
quality_report.loc["nsm", "non_integer_values"] = (df["nsm"].dropna() % 1 != 0).sum()
quality_report.loc["nsm", "values_above_86400"] = (df["nsm"] > 86_400).sum()
quality_report


## 4. Parse the timestamp and create time-based features

The date column is currently stored as text. Converting it to datetime enables sorting, time-based splitting, and more informative features such as hour, day-of-week, and weekend status.

In [ ]:
# Parse dates using the exact format used in the file
df["date_time"] = pd.to_datetime(df["date_time"], format="%d-%m-%Y %H:%M")

# Sort chronologically
df = df.sort_values("date_time").reset_index(drop=True)

# Convert numeric columns safely
for col in [
    "usage_kwh",
    "lagging_reactive_kvarh",
    "leading_reactive_kvarh",
    "co2_tco2",
    "lagging_power_factor",
    "leading_power_factor",
    "nsm",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fix invalid values using domain rules
df.loc[df["lagging_power_factor"] > 100, "lagging_power_factor"] = np.nan
df.loc[df["leading_power_factor"] > 100, "leading_power_factor"] = np.nan
df.loc[df["nsm"] > 86_400, "nsm"] = np.nan

# Time-based features that often matter for load classification
df["hour"] = df["date_time"].dt.hour
df["month"] = df["date_time"].dt.month
df["day_of_week"] = df["date_time"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["nsm_from_time"] = df["date_time"].dt.hour * 3600 + df["date_time"].dt.minute * 60

df[["date_time", "hour", "month", "day_of_week", "is_weekend", "nsm_from_time"]].head()

## 5. Create a realistic time-based train/test split

A random split would leak future information into training. This project explicitly requires the last month to act as the test set.

In [ ]:
train_cutoff = pd.Timestamp("2018-12-01")

train = df[df["date_time"] < train_cutoff].copy()
test = df[df["date_time"] >= train_cutoff].copy()

print(f"Train rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"Train date range: {train['date_time'].min()} to {train['date_time'].max()}")
print(f"Test date range: {test['date_time'].min()} to {test['date_time'].max()}")

target = "load_type"
feature_cols = [
    "usage_kwh", "lagging_reactive_kvarh", "leading_reactive_kvarh", "co2_tco2",
    "lagging_power_factor", "leading_power_factor", "nsm", "hour", "month", "day_of_week",
    "is_weekend", "nsm_from_time",
]

X_train = train[feature_cols]
y_train = train[target]
X_test = test[feature_cols]
y_test = test[target]

print("Target distribution in train:")
print(y_train.value_counts())
print("\nTarget distribution in test:")
print(y_test.value_counts())

## 6. Train and compare models

We use a pipeline to handle missing values and scaling consistently. Logistic regression and random forest are good baselines for this task.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            feature_cols,
        )
    ]
)

models = {
    "logistic_regression": LogisticRegression(max_iter=5000, multi_class="auto", random_state=42),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1,
    ),
}

results = []
for model_name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    })

results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", models[best_model_name]),
])
best_pipeline.fit(X_train, y_train)
best_pred = best_pipeline.predict(X_test)

print(f"Best model: {best_model_name}")
print("\nClassification report:")
print(classification_report(y_test, best_pred, digits=4, zero_division=0))

cm = confusion_matrix(y_test, best_pred, labels=["Light_Load", "Medium_Load", "Maximum_Load"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_title(f"Confusion Matrix ({best_model_name})")
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(["Light_Load", "Medium_Load", "Maximum_Load"])
ax.set_yticklabels(["Light_Load", "Medium_Load", "Maximum_Load"])
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 7. Final notes

This revision improves the workflow in several ways:
- It keeps the analysis reproducible and easier to follow.
- It validates the data before model training.
- It uses a proper time-based split so the December data acts as a true future test set.
- It creates time features from the timestamp, which are likely relevant in power-system load forecasting.
- It compares multiple models with consistent preprocessing and evaluation.

If this were to be extended further, the next natural steps would be:
- tune hyperparameters with cross-validation on the training data only
- experiment with gradient boosting models
- save the trained model and metrics for reporting
- test whether lagged or rolling features improve temporal performance.